# Using Timelink in notebooks: tutorial

> First time use: follow instructions in the [README.md](README.md) file in this directory.

In [ ]:
import timelink
timelink.version

## Initial setup

Create a TimelinkNotebook object. This will be the interface for most of your interactions with Timelink.

The first time you run this notebook, you will be prompted to install ìpykernel. This is required to run Timelink in a notebook.

Note:
* First time run takes a little time as the required Docker images are downloaded.
* Timelink will default to using sqlite as the database, see [Receipts notebook for more control](1-receipts.ipynb)


In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()


TimelinkNotebook created a database to store data and a Kleio server to 
translate transcriptions in Kleio notation into data that can be imported
into a database.

To get information about the database and the Kleio server do ```tnlb.print_info()```

In [ ]:
tlnb.print_info()

### Database status

Count the number of rows in each table in the database.


In [ ]:
tlnb.table_row_count_df()

### Show the kleio files available 

In [ ]:
kleio_files = tlnb.get_kleio_files()
kleio_files

## Optionally clean translations 

In [ ]:
tlnb.kleio_server.translation_clean("", recurse="yes")


# Update the database from Kleio sources.

Updates source translations and imports into database sources with no errors.

Only changed files since last import, or new files, will be processed.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

tlnb.update_from_sources()

## Check the status of the files

Check the import status of the translated files

I: Imported

E: Imported with error

W: Imported with warnings no errors

N: Not imported

U: Translation updated need to reimport

In [ ]:
imported_files_df = tlnb.get_import_status()
imported_files_df[["import_status","import_errors","import_warnings","name","errors","warnings","path"]].sort_values("name")


### Check the translation report

In [ ]:
rpt = tlnb.get_translation_report(imported_files_df, 12)
print(rpt)


### Get the import report for a file

In case there are errors in the import phase check the import report.

In [ ]:
rpt = tlnb.get_import_rpt(imported_files_df, rows=[13])
print(rpt)

## Todo

This as data frame in a single function
* TimelinkNotebook.translate([files_df,paths=List[str], rows=List[int],status="T"])
* TimelinkNotebook.import([files_df,paths=List[str], rows=List[int],status="V"])



# Obter dados /  Getting data

### Procurar pessoa, mostrar em notação Kleio

---
### Search for people, show in Kleio notation

In [ ]:
from timelink.api.models import Person

show_only=10

with tlnb.db.session() as session:
    ricci = session.query(Person).filter(Person.name.like('%Ricci')).all()

    for mr in ricci[:show_only]:
        print(mr.to_kleio())
        print()

In [ ]:
from sqlalchemy import select

from timelink.api.models import Person, Attribute

show_only=10

with tlnb.db.session() as session:
    stmt = select(Person).join(Person.attributes.and_(Attribute.the_type == 'estadia',
                                                      Attribute.the_value.like('Pequim')))
    print(stmt)
    persons = session.execute(stmt).scalars().all()

    for person in persons[:show_only]:
        p = tlnb.db.get_person(person.id)
        print(p.to_kleio())




###  Obter um dataframe a partir de atributos

---


###  Get a Dataframe from attributes


#### Exemplo: Missionários que estiveram em Pequim, onde entraram
na ordem, e que nacionalidade tinham.

In [ ]:
from timelink.pandas import entities_with_attribute


# Get list of people with with a certain value in a specific attribute
df = entities_with_attribute(
                    entity_type='person',
                    the_type='estadia',
                    the_value='Pequim',
                    more_info=['name'],
                    more_cols=['jesuita-entrada','nacionalidade'],
                    db=tlnb.db,
                    sql_echo=False)
df.info()

In [ ]:
df.head(55).sort_values('jesuita-entrada')



###  Remover colunas sem valores

---



###  Remove empty columns

In [ ]:
df.dropna(how='all', axis=1, inplace=True)
df.info()

In [ ]:
df.head(5)


## Contagens

---

## Counting



###  Contagem de atributos a partir de uma tabela em memória

---

###  Count attributes from an existing dataframe



In [ ]:
# create a column with the index values which are the id numbers
df['id'] = df.index.values

col = 'jesuita-entrada' # subotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = df.groupby(col).agg({'id':'nunique',
                                                  'estadia':'min',
                                                  'estadia':'max'})

df_totals.sort_values('id',ascending= False).head(30)


### Contagens na base de dados

Quando o atributo tem muitos valores e não é necessário
ter todas as pessoas em memória: contagem feita na base de dados

---

### Counting directly in the database
When there are many values and it is not
necessary to have all the people in memory:
count directly in the database.




In [ ]:
from timelink.pandas import attribute_values

df_totals = attribute_values('estadia', dates_between=("1580-00-00","1610-00-00"),db=tlnb.db)
df_totals.head(20)

#### Filtrar por datas

---

#### Filter by dates



In [ ]:
df_totals = attribute_values('jesuita-entrada',dates_between=('1580','1600'),db=tlnb.db)

In [ ]:
df_totals.head(10)

## Visualizar registos

---

## View records





### Visualizar uma pessoa

---

### View a person


#### Atributos de uma pessoa numa tabela, uma linha por attributo

---

#### Person attributes in a dataframe, one line per attribute

In [ ]:
import pandas as pd
from timelink.pandas import group_attributes as person_attributes

pd.set_option('display.max_rows',1000)

id = 'deh-matteo-ricci'
pdf = person_attributes([id],db=tlnb.db)  # note id in a list
pdf[['date','type','value','attr_obs']].sort_values(['date','type'])

In [ ]:
p = tlnb.db.get_person(id)
print(p.to_kleio())

#### Atributos de uma pessoa numa tabela, attributos em colunas

---

#### Person attributes in a dataframe, attributes in columns

In [ ]:
# Get list of people with with a certain value in a specific attribute

id = 'deh-matteo-ricci'  # matheo ricci
# id = 'deh-antonio-de-magalhaes-ref1'  # Alexandre Metelo de Sousa
df = entities_with_attribute(
                    entity_type='person',
                    the_type='estadia',  # we need a base attribute
                    the_value='Pequim',
                    more_info=['name'],
                    more_cols=['jesuita-entrada','jesuita-estatuto','morte'],
                    dates_in=('1580','1610'),
                    db=tlnb.db,
                    sql_echo=False)
view_cols = ['name','estadia','estadia.date','morte.date','jesuita-entrada','jesuita-entrada.date']
df.info()
df[view_cols].sort_values('estadia.date')

#### Examinar potenciais duplicados

---

#### Examine potentital duplicates

In [ ]:
from timelink.pandas.group_attributes import display_group_attributes
pd.set_option('display.max_rows',250)

no_show=['dehergne']

dup_ids = ricci.index.unique()  #

display_group_attributes(dup_ids,
                             header_cols=['estadia'],
                             exclude_attributes=no_show,
                             sort_attributes=['date','type','value'],
                             cmap_name='Pastel1')